# Statistical Analysis Script for Energy Consumption Study

Performs comprehensive analysis on energy consumption data across frameworks

## Analysis includes:
1. Descriptive statistics (mean, SD)
2. Shapiro-Wilk normality test
3. Kruskal-Wallis test for framework differences
4. Mann-Whitney U with Bonferroni correction (post-hoc)
5. Spearman correlation (energy vs. performance)
6. Publication-quality plots (boxplots, scaling curves, scatterplots)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, kruskal, mannwhitneyu, spearmanr
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9


In [ ]:
FRAMEWORKS = ['react', 'vue', 'angular', 'svelte']
SCENARIOS = ['simple', 'medium', 'complex']
MEASUREMENTS_DIR = Path('../measurements')
OUTPUT_DIR = Path('.')
ALPHA = 0.05


In [ ]:
def load_data():
    """
    Load measurement data from JSON files
    Expected format: One JSON object per line
    """
    data = []
    
    for framework in FRAMEWORKS:
        for scenario in SCENARIOS:
            json_file = MEASUREMENTS_DIR / framework / f'{scenario}.json'
            if json_file.exists():
                with open(json_file, 'r') as f:
                    for line in f:
                        if line.strip():
                            try:
                                record = json.loads(line)
                                record['framework'] = framework
                                record['scenario'] = scenario
                                data.append(record)
                            except json.JSONDecodeError:
                                continue
    
    return pd.DataFrame(data)



## Data Loading Functions

In [ ]:
def extract_metrics(df):
    """Extract energy and performance metrics from data"""
    metrics = []
    
    for _, row in df.iterrows():
        metric = {
            'framework': row['framework'],
            'scenario': row['scenario'],
            'timestamp': row.get('timestamp', ''),
            'total_duration': row.get('totalDuration', 0)
        }
        
        # Extract energy metrics (if available from RAPL/Scaphandre)
        # Adjust field names based on actual measurement tool output
        metric['energy_joules'] = row.get('energy', row.get('energy_joules', np.nan))
        metric['peak_power_watts'] = row.get('peak_power', row.get('peak_power_watts', np.nan))
        
        # Extract performance metrics from actions
        actions = row.get('actions', [])
        for action in actions:
            if action.get('action') == 'total':
                metric['total_duration'] = action.get('duration', metric['total_duration'])
        
        metrics.append(metric)
    
    return pd.DataFrame(metrics)



In [ ]:
def descriptive_statistics(df):
    """Calculate descriptive statistics"""
    print("\n" + "="*60)
    print("DESCRIPTIVE STATISTICS")
    print("="*60)
    
    desc_stats = df.groupby(['framework', 'scenario']).agg({
        'energy_joules': ['mean', 'std', 'min', 'max', 'median'],
        'total_duration': ['mean', 'std', 'min', 'max', 'median']
    }).round(2)
    
    print(desc_stats)
    
    # Save to CSV
    desc_stats.to_csv(OUTPUT_DIR / 'descriptive_statistics.csv')
    print(f"\nSaved to: {OUTPUT_DIR / 'descriptive_statistics.csv'}")
    
    return desc_stats



## Statistical Analysis Functions

In [ ]:
def normality_tests(df):
    """Perform Shapiro-Wilk normality tests"""
    print("\n" + "="*60)
    print("NORMALITY TESTS (Shapiro-Wilk)")
    print("="*60)
    
    results = []
    
    for framework in FRAMEWORKS:
        for scenario in SCENARIOS:
            subset = df[(df['framework'] == framework) & (df['scenario'] == scenario)]
            if len(subset) >= 3:  # Minimum for Shapiro-Wilk
                energy_data = subset['energy_joules'].dropna()
                duration_data = subset['total_duration'].dropna()
                
                if len(energy_data) >= 3:
                    stat_e, p_e = shapiro(energy_data)
                    results.append({
                        'framework': framework,
                        'scenario': scenario,
                        'metric': 'energy_joules',
                        'statistic': stat_e,
                        'p_value': p_e,
                        'normal': p_e > ALPHA
                    })
                
                if len(duration_data) >= 3:
                    stat_d, p_d = shapiro(duration_data)
                    results.append({
                        'framework': framework,
                        'scenario': scenario,
                        'metric': 'total_duration',
                        'statistic': stat_d,
                        'p_value': p_d,
                        'normal': p_d > ALPHA
                    })
    
    norm_df = pd.DataFrame(results)
    print(norm_df.to_string(index=False))
    norm_df.to_csv(OUTPUT_DIR / 'normality_tests.csv', index=False)
    
    return norm_df



In [ ]:
def kruskal_wallis_test(df, metric='energy_joules'):
    """Perform Kruskal-Wallis test for framework differences"""
    print(f"\n" + "="*60)
    print(f"KRUSKAL-WALLIS TEST ({metric})")
    print("="*60)
    
    results = []
    
    for scenario in SCENARIOS:
        scenario_data = df[df['scenario'] == scenario]
        groups = [scenario_data[scenario_data['framework'] == f][metric].dropna().values 
                  for f in FRAMEWORKS 
                  if len(scenario_data[scenario_data['framework'] == f][metric].dropna()) > 0]
        
        if len(groups) >= 2:
            stat, p_value = kruskal(*groups)
            results.append({
                'scenario': scenario,
                'statistic': stat,
                'p_value': p_value,
                'significant': p_value < ALPHA
            })
            print(f"\n{scenario.capitalize()} scenario:")
            print(f"  H-statistic: {stat:.4f}")
            print(f"  p-value: {p_value:.6f}")
            print(f"  Significant difference: {'Yes' if p_value < ALPHA else 'No'}")
    
    kw_df = pd.DataFrame(results)
    kw_df.to_csv(OUTPUT_DIR / f'kruskal_wallis_{metric}.csv', index=False)
    
    return kw_df



In [ ]:
def mann_whitney_posthoc(df, metric='energy_joules'):
    """Mann-Whitney U post-hoc tests with Bonferroni correction"""
    print(f"\n" + "="*60)
    print(f"MANN-WHITNEY U POST-HOC TESTS ({metric})")
    print("="*60)
    
    results = []
    
    for scenario in SCENARIOS:
        scenario_data = df[df['scenario'] == scenario]
        
        # All pairwise comparisons
        comparisons = []
        for i, f1 in enumerate(FRAMEWORKS):
            for f2 in FRAMEWORKS[i+1:]:
                group1 = scenario_data[scenario_data['framework'] == f1][metric].dropna()
                group2 = scenario_data[scenario_data['framework'] == f2][metric].dropna()
                
                if len(group1) > 0 and len(group2) > 0:
                    comparisons.append((f1, f2))
        
        # Bonferroni correction
        n_comparisons = len(comparisons)
        corrected_alpha = ALPHA / n_comparisons if n_comparisons > 0 else ALPHA
        
        print(f"\n{scenario.capitalize()} scenario:")
        print(f"  Number of comparisons: {n_comparisons}")
        print(f"  Corrected alpha (Bonferroni): {corrected_alpha:.6f}")
        
        for f1, f2 in comparisons:
            group1 = scenario_data[scenario_data['framework'] == f1][metric].dropna()
            group2 = scenario_data[scenario_data['framework'] == f2][metric].dropna()
            
            stat, p_value = mannwhitneyu(group1, group2, alternative='two-sided')
            significant = p_value < corrected_alpha
            
            results.append({
                'scenario': scenario,
                'framework1': f1,
                'framework2': f2,
                'statistic': stat,
                'p_value': p_value,
                'corrected_alpha': corrected_alpha,
                'significant': significant
            })
            
            sig_str = '***' if significant else ''
            print(f"  {f1} vs {f2}: U={stat:.2f}, p={p_value:.6f} {sig_str}")
    
    mw_df = pd.DataFrame(results)
    mw_df.to_csv(OUTPUT_DIR / f'mann_whitney_posthoc_{metric}.csv', index=False)
    
    return mw_df



In [ ]:
def correlation_analysis(df):
    """Spearman correlation between energy and performance"""
    print("\n" + "="*60)
    print("SPEARMAN CORRELATION (Energy vs. Performance)")
    print("="*60)
    
    results = []
    
    for framework in FRAMEWORKS:
        for scenario in SCENARIOS:
            subset = df[(df['framework'] == framework) & (df['scenario'] == scenario)]
            energy = subset['energy_joules'].dropna()
            duration = subset['total_duration'].dropna()
            
            if len(energy) >= 3 and len(duration) >= 3:
                # Align indices
                common_idx = energy.index.intersection(duration.index)
                if len(common_idx) >= 3:
                    corr, p_value = spearmanr(energy.loc[common_idx], duration.loc[common_idx])
                    
                    results.append({
                        'framework': framework,
                        'scenario': scenario,
                        'correlation': corr,
                        'p_value': p_value,
                        'significant': p_value < ALPHA
                    })
                    
                    sig_str = '***' if p_value < ALPHA else ''
                    print(f"{framework} - {scenario}: rho={corr:.4f}, p={p_value:.6f} {sig_str}")
    
    corr_df = pd.DataFrame(results)
    corr_df.to_csv(OUTPUT_DIR / 'correlation_analysis.csv', index=False)
    
    return corr_df



In [ ]:
def plot_boxplots(df, metric='energy_joules', title_suffix='Energy Consumption'):
    """Create boxplots for framework comparison"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, scenario in enumerate(SCENARIOS):
        scenario_data = df[df['scenario'] == scenario]
        
        data_for_plot = []
        labels = []
        for framework in FRAMEWORKS:
            values = scenario_data[scenario_data['framework'] == framework][metric].dropna()
            if len(values) > 0:
                data_for_plot.append(values)
                labels.append(framework.capitalize())
        
        if data_for_plot:
            bp = axes[idx].boxplot(data_for_plot, labels=labels, patch_artist=True)
            
            # Color coding
            colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
            for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            axes[idx].set_title(f'{scenario.capitalize()} Scenario')
            axes[idx].set_ylabel(f'{title_suffix}')
            axes[idx].grid(True, alpha=0.3)
            axes[idx].tick_params(axis='x', rotation=45)
    
    plt.suptitle(f'Framework Comparison: {title_suffix}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'01_energy_boxplot.png', bbox_inches='tight')
    print(f"\nSaved plot: {OUTPUT_DIR / '01_energy_boxplot.png'}")



## Plotting Functions

In [ ]:
def plot_scaling_curves(df, metric='energy_joules'):
    """Create scaling curves showing how metrics change with workload"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    scenario_order = {'simple': 100, 'medium': 1000, 'complex': 5000}
    
    for framework in FRAMEWORKS:
        x_values = []
        y_values = []
        y_errors = []
        
        for scenario in SCENARIOS:
            subset = df[(df['framework'] == framework) & (df['scenario'] == scenario)]
            values = subset[metric].dropna()
            
            if len(values) > 0:
                x_values.append(scenario_order[scenario])
                y_values.append(values.mean())
                y_errors.append(values.std() if len(values) > 1 else 0)
        
        if x_values:
            ax.errorbar(x_values, y_values, yerr=y_errors, 
                       marker='o', label=framework.capitalize(), 
                       linewidth=2, markersize=8, capsize=5)
    
    ax.set_xlabel('Number of Items', fontweight='bold')
    ax.set_ylabel(f'{metric.replace("_", " ").title()}', fontweight='bold')
    ax.set_title('Scaling Behavior Across Workloads', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'02_scaling_graph.png', bbox_inches='tight')
    print(f"Saved plot: {OUTPUT_DIR / '02_scaling_graph.png'}")



In [ ]:
def plot_correlation(df):
    """Scatterplot showing energy vs. performance correlation"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, scenario in enumerate(SCENARIOS):
        scenario_data = df[df['scenario'] == scenario]
        
        for framework in FRAMEWORKS:
            subset = scenario_data[scenario_data['framework'] == framework]
            energy = subset['energy_joules'].dropna()
            duration = subset['total_duration'].dropna()
            
            common_idx = energy.index.intersection(duration.index)
            if len(common_idx) >= 2:
                axes[idx].scatter(duration.loc[common_idx], energy.loc[common_idx],
                                label=framework.capitalize(), alpha=0.6, s=50)
        
        axes[idx].set_xlabel('Execution Time (ms)', fontweight='bold')
        axes[idx].set_ylabel('Energy (J)', fontweight='bold')
        axes[idx].set_title(f'{scenario.capitalize()} Scenario', fontweight='bold')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
    
    plt.suptitle('Energy vs. Performance Correlation', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'03_correlation.png', bbox_inches='tight')
    print(f"Saved plot: {OUTPUT_DIR / '03_correlation.png'}")



In [ ]:
def plot_framework_comparison(df, metric='energy_joules'):
    """Bar chart comparing frameworks across scenarios"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(SCENARIOS))
    width = 0.2
    
    for i, framework in enumerate(FRAMEWORKS):
        means = []
        stds = []
        for scenario in SCENARIOS:
            subset = df[(df['framework'] == framework) & (df['scenario'] == scenario)]
            values = subset[metric].dropna()
            means.append(values.mean() if len(values) > 0 else 0)
            stds.append(values.std() if len(values) > 1 else 0)
        
        ax.bar(x + i * width, means, width, yerr=stds, 
               label=framework.capitalize(), alpha=0.8, capsize=5)
    
    ax.set_xlabel('Scenario', fontweight='bold')
    ax.set_ylabel(f'{metric.replace("_", " ").title()}', fontweight='bold')
    ax.set_title('Framework Comparison Across Scenarios', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([s.capitalize() for s in SCENARIOS])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'04_framework_comparison.png', bbox_inches='tight')
    print(f"Saved plot: {OUTPUT_DIR / '04_framework_comparison.png'}")



In [ ]:
def plot_statistical_significance(df, metric='energy_joules'):
    """Heatmap showing statistical significance between frameworks"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, scenario in enumerate(SCENARIOS):
        scenario_data = df[df['scenario'] == scenario]
        
        # Create comparison matrix
        comparison_matrix = np.zeros((len(FRAMEWORKS), len(FRAMEWORKS)))
        
        for i, f1 in enumerate(FRAMEWORKS):
            for j, f2 in enumerate(FRAMEWORKS):
                if i != j:
                    group1 = scenario_data[scenario_data['framework'] == f1][metric].dropna()
                    group2 = scenario_data[scenario_data['framework'] == f2][metric].dropna()
                    
                    if len(group1) > 0 and len(group2) > 0:
                        _, p_value = mannwhitneyu(group1, group2, alternative='two-sided')
                        comparison_matrix[i, j] = -np.log10(p_value + 1e-10)  # -log10(p)
        
        im = axes[idx].imshow(comparison_matrix, cmap='RdYlGn', aspect='auto')
        axes[idx].set_xticks(range(len(FRAMEWORKS)))
        axes[idx].set_yticks(range(len(FRAMEWORKS)))
        axes[idx].set_xticklabels([f.capitalize() for f in FRAMEWORKS])
        axes[idx].set_yticklabels([f.capitalize() for f in FRAMEWORKS])
        axes[idx].set_title(f'{scenario.capitalize()} Scenario', fontweight='bold')
        
        plt.colorbar(im, ax=axes[idx], label='-log10(p-value)')
    
    plt.suptitle('Statistical Significance Between Frameworks', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'05_significance_heatmap.png', bbox_inches='tight')
    print(f"Saved plot: {OUTPUT_DIR / '05_significance_heatmap.png'}")



In [ ]:
def generate_summary_report(df):
    """Generate a comprehensive summary report"""
    report = []
    report.append("="*60)
    report.append("ENERGY CONSUMPTION STUDY - SUMMARY REPORT")
    report.append("="*60)
    report.append("")
    
    # Overview
    report.append("OVERVIEW")
    report.append("-"*60)
    report.append(f"Frameworks tested: {', '.join([f.capitalize() for f in FRAMEWORKS])}")
    report.append(f"Scenarios: {', '.join([s.capitalize() for s in SCENARIOS])}")
    report.append(f"Total measurements: {len(df)}")
    report.append("")
    
    # Key findings
    report.append("KEY FINDINGS")
    report.append("-"*60)
    
    for scenario in SCENARIOS:
        scenario_data = df[df['scenario'] == scenario]
        if len(scenario_data) > 0:
            best_framework = scenario_data.groupby('framework')['energy_joules'].mean().idxmin()
            worst_framework = scenario_data.groupby('framework')['energy_joules'].mean().idxmax()
            report.append(f"{scenario.capitalize()}: Most efficient = {best_framework.capitalize()}, "
                         f"Least efficient = {worst_framework.capitalize()}")
    
    report.append("")
    
    # Save report
    report_text = "\n".join(report)
    with open(OUTPUT_DIR / 'summary_report.txt', 'w') as f:
        f.write(report_text)
    
    print(report_text)
    print(f"\nFull report saved to: {OUTPUT_DIR / 'summary_report.txt'}")



In [ ]:
def main():
    """Main analysis pipeline"""
    print("Starting Energy Consumption Analysis...")
    print("="*60)
    
    # Load data
    print("\nLoading data...")
    raw_df = load_data()
    
    if len(raw_df) == 0:
        print("ERROR: No data found. Please ensure measurement files exist.")
        print(f"Expected location: {MEASUREMENTS_DIR}")
        return
    
    # Extract metrics
    df = extract_metrics(raw_df)
    print(f"Loaded {len(df)} measurements")
    
    # Handle missing energy data (create mock data for demonstration)
    if df['energy_joules'].isna().all():
        print("\nWarning: No energy data found. Generating synthetic data for demonstration.")
        # Synthetic energy data proportional to duration
        df['energy_joules'] = df['total_duration'] * (np.random.uniform(0.5, 1.5, len(df)) * 0.1)
        df['peak_power_watts'] = df['energy_joules'] / (df['total_duration'] / 1000)  # W = J/s
    
    # Perform analyses
    descriptive_statistics(df)
    normality_tests(df)
    kruskal_wallis_test(df, 'energy_joules')
    mann_whitney_posthoc(df, 'energy_joules')
    correlation_analysis(df)
    
    # Generate plots
    print("\n" + "="*60)
    print("GENERATING PLOTS")
    print("="*60)
    plot_boxplots(df, 'energy_joules', 'Energy Consumption (J)')
    plot_scaling_curves(df, 'energy_joules')
    plot_correlation(df)
    plot_framework_comparison(df, 'energy_joules')
    plot_statistical_significance(df, 'energy_joules')
    
    # Generate summary
    generate_summary_report(df)
    
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)
    print(f"\nAll outputs saved to: {OUTPUT_DIR}")



In [ ]:
# Execute main analysis
main()

## Main Analysis Pipeline

Run the main function to execute the complete analysis: